In [0]:
#Load path of json files organized by patch
path_json_matches = "/Volumes/workspace/bronze/raw_data/*/matches/"
path_json_timelines = "/Volumes/workspace/bronze/raw_data/*/timelines/"

#Load Json to DataFrame
df_matches = spark.read.format("json").load(path_json_matches)
df_timelines = spark.read.format("json").load(path_json_timelines)

#Create views for SQL
df_matches.createOrReplaceTempView("new_matches")
df_timelines.createOrReplaceTempView("new_timelines")

In [0]:
%sql
USE CATALOG workspace;
USE SCHEMA bronze;

--Create a table if it doesn't exist and Merge new data in Bronze
CREATE TABLE IF NOT EXISTS b_matches
USING delta
AS SELECT * FROM new_matches
WHERE 1=0;

MERGE INTO b_matches
USING new_matches
ON b_matches.metadata.matchId = new_matches.metadata.matchId
WHEN NOT MATCHED THEN INSERT *;

CREATE TABLE IF NOT EXISTS b_timelines
USING delta
AS SELECT * FROM new_timelines
WHERE 1=0;

MERGE INTO b_timelines
USING new_timelines
ON b_timelines.metadata.matchId = new_timelines.metadata.matchId
WHEN NOT MATCHED THEN INSERT *;

In [0]:
%sql
-- Checking the last 3 patches and deleting the previous ones

USE CATALOG workspace;
USE SCHEMA bronze;

CREATE OR REPLACE TEMP VIEW patches_to_keep AS
SELECT patch FROM (
    SELECT DISTINCT
        concat(split(info.gameVersion, '\\.')[0], '.', split(info.gameVersion, '\\.')[1]) AS patch,
        CAST(split(info.gameVersion, '\\.')[0] AS INT) AS major,
        CAST(split(info.gameVersion, '\\.')[1] AS INT) AS minor
    FROM b_matches
)
ORDER BY major DESC, minor DESC
LIMIT 3;  -- number of patches to keep, same as sync_to_databricks.py

-- Deleting old matches
DELETE FROM b_matches
WHERE concat(split(info.gameVersion, '\\.')[0], '.', split(info.gameVersion, '\\.')[1])
      NOT IN (SELECT patch FROM patches_to_keep);

-- Deleting the linked timelines
DELETE FROM b_timelines
WHERE metadata.matchId NOT IN (SELECT metadata.matchId FROM b_matches);